# TP 05-B — Emojifier: GloVe + LSTM

**Course:** Natural Language Processing  
**Session:** 5 / 8  
**Submission deadline:** one week after the session

---

## Context

In TP-04 you used GloVe mean pooling to classify AG News articles.
In TP-05A you trained an LSTM from scratch on spam detection.

This TP combines both ideas:
- Use **pre-trained GloVe embeddings** (frozen, loaded via gensim)
- Feed them into an **LSTM** that respects word order

**Task:** given a short sentence, predict the most appropriate emoji (5 classes).

| Label | Emoji | Examples |
|-------|-------|----------|
| 0 | ❤️ | "I love you", "You mean the world to me" |
| 1 | ⚽ | "Let us play football", "Great match" |
| 2 | 😂 | "That is hilarious", "I laughed until I cried" |
| 3 | 😞 | "I feel really down", "I failed the exam" |
| 4 | 🍴 | "Let us eat", "That meal was delicious" |

**Two models** — you build and compare both:
- **V1 — Emojifier-V1**: GloVe mean pool → Dense (no word order, like TP-04)
- **V2 — Emojifier-V2**: GloVe frozen embedding → LSTM (word order preserved)

---

## Roadmap

| Part | Task |
|------|------|
| 1 | Load dataset and GloVe |
| 2 | Emojifier-V1: GloVe mean pool + softmax |
| 3 | Emojifier-V2: frozen GloVe + LSTM |
| 4 | Compare V1 vs V2 |
| 5 (bonus) | Test on your own sentences |

---


## Setup


In [ ]:
# Uncomment to install on Colab
# !pip install tensorflow scikit-learn matplotlib gensim -q

import re
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

import tensorflow as tf
from tensorflow.keras.models import Model
from tensorflow.keras.layers import (
    Input, Dense, Dropout, LSTM, Embedding, GlobalAveragePooling1D
)
from tensorflow.keras.preprocessing.sequence import pad_sequences
from sklearn.metrics import classification_report, confusion_matrix, ConfusionMatrixDisplay

RANDOM_STATE = 42
tf.random.set_seed(RANDOM_STATE)
np.random.seed(RANDOM_STATE)

GLOVE_DIM  = 50       # 50 | 100 | 300 -- must match the gensim model
MAX_LEN    = 10       # sentences are short; 10 words is enough
NUM_CLASSES = 5
EPOCHS     = 50
BATCH_SIZE = 32

LABEL_EMOJIS = {0: '❤️', 1: '⚽', 2: '😂', 3: '😞', 4: '🍴'}

print(f'TensorFlow: {tf.__version__}')
print('Setup OK.')


---
## Part 1 — Dataset and GloVe


### 1.1 — Load the dataset


In [ ]:
TRAIN_DATA = [
    ("I love you", 0), ("I adore you", 0), ("Sending you my love", 0),
    ("I miss you so much", 0), ("You are my heart", 0), ("I cherish every moment with you", 0),
    ("Forever and always", 0), ("You mean the world to me", 0), ("My love for you is endless", 0),
    ("You complete me", 0), ("Together we are stronger", 0), ("I am so proud of you", 0),
    ("You make me so happy", 0), ("I care about you deeply", 0), ("My favourite person", 0),
    ("You are my sunshine", 0), ("I love spending time with you", 0), ("You are wonderful", 0),
    ("I appreciate everything you do", 0), ("You have my heart", 0),
    ("I will always be here for you", 0), ("You are so special to me", 0),
    ("I love you to the moon and back", 0), ("You are my everything", 0),
    ("I think about you all the time", 0), ("Best day ever with you", 0),
    ("Love is all around", 0), ("You make my heart sing", 0),
    ("Grateful to have you in my life", 0), ("You light up my world", 0),
    ("Let us play", 1), ("Let us go to the gym", 1), ("Train hard", 1),
    ("He scored a goal", 1), ("Best game ever", 1), ("Go team go", 1),
    ("Let us play football", 1), ("Play cricket with us", 1), ("Great match last night", 1),
    ("I enjoy running every morning", 1), ("Hit the gym after work", 1),
    ("Let us do some exercise", 1), ("The tournament starts tomorrow", 1),
    ("She is the best player on the team", 1), ("We won the championship", 1),
    ("Practice makes perfect", 1), ("Let us shoot some hoops", 1),
    ("The referee made a bad call", 1), ("Extra time and penalties", 1),
    ("Run faster", 1), ("Swimming is great exercise", 1), ("Let us go for a run", 1),
    ("The team is training hard", 1), ("He hit a home run", 1),
    ("She broke the world record", 1), ("Time to work out", 1),
    ("The stadium was full of fans", 1), ("He tackled brilliantly", 1),
    ("She won the race by a mile", 1), ("Great teamwork today", 1),
    ("This is so funny", 2), ("That joke made me laugh", 2),
    ("You are so silly", 2), ("Ha ha ha", 2), ("I cannot stop laughing", 2),
    ("That is hilarious", 2), ("He always makes me laugh", 2),
    ("Too funny for words", 2), ("She had the whole room laughing", 2),
    ("I am dying of laughter", 2), ("What a ridiculous situation", 2),
    ("That pun was terrible and I loved it", 2), ("You crack me up every time", 2),
    ("Best comedian I have ever seen", 2), ("I nearly fell off my chair laughing", 2),
    ("Send me more funny videos", 2), ("That sketch was pure comedy gold", 2),
    ("He tripped and I could not help laughing", 2),
    ("The bloopers at the end were the best part", 2), ("I laughed until I cried", 2),
    ("This meme is perfect", 2), ("Comedy genius", 2),
    ("She does the best impressions", 2), ("I snorted from laughing so hard", 2),
    ("The outtakes are better than the film", 2), ("Your laugh is contagious", 2),
    ("That was the funniest thing all week", 2),
    ("I cannot keep a straight face around you", 2),
    ("He always has a joke ready", 2), ("We laughed all the way home", 2),
    ("I am not happy", 3), ("I am upset", 3), ("This is terrible news", 3),
    ("She is my worst nightmare", 3), ("This is so sad", 3),
    ("I feel really down today", 3), ("Everything is going wrong", 3),
    ("I miss those better days", 3), ("Nothing ever goes my way", 3),
    ("I cannot believe that happened", 3), ("I am devastated", 3),
    ("She broke my heart", 3), ("That news really hurt", 3),
    ("I have been crying all day", 3), ("Feeling lonely tonight", 3),
    ("I failed the exam", 3), ("Nobody understands how I feel", 3),
    ("Today was really hard", 3), ("I am exhausted and discouraged", 3),
    ("Things are not going well", 3), ("I feel let down", 3),
    ("It was a really rough week", 3), ("I am losing hope", 3),
    ("She left without saying goodbye", 3), ("I regret so many things", 3),
    ("Feeling hopeless right now", 3), ("The worst day in a long time", 3),
    ("I just want to be alone", 3),
    ("Let us eat", 4), ("She is a good cook", 4), ("I am hungry", 4),
    ("What is for dinner", 4), ("Let us grab a bite", 4),
    ("That meal was delicious", 4), ("I love trying new restaurants", 4),
    ("Best pizza I have ever had", 4), ("Time for lunch", 4),
    ("She baked an amazing cake", 4), ("The menu looks incredible", 4),
    ("I could eat pasta every day", 4), ("Breakfast is the best meal", 4),
    ("Let us have sushi tonight", 4), ("The food here is always fresh", 4),
    ("I am craving chocolate right now", 4), ("He is a brilliant chef", 4),
    ("That soup warmed me right up", 4), ("Cooking is my favourite hobby", 4),
    ("Let us try that new place downtown", 4), ("I made pancakes this morning", 4),
    ("The dessert was the best part", 4), ("Tacos for dinner again", 4),
    ("She makes the best roast chicken", 4),
    ("Nothing beats a home cooked meal", 4), ("I am full but I want more", 4),
    ("That smoothie was so refreshing", 4),
]

TEST_DATA  = [
    ("I love you so much", 0), ("You are my love", 0), ("I miss you", 0),
    ("My heart belongs to you", 0), ("You are my true love", 0),
    ("Always in my heart", 0), ("I will love you forever", 0),
    ("You are precious to me", 0), ("My dearest friend", 0),
    ("Together forever", 0), ("You are so dear to me", 0),
    ("Lets exercise", 1), ("He plays well", 1), ("She loves swimming", 1),
    ("Run to win", 1), ("Score a goal", 1), ("Great penalty", 1),
    ("Let us cycle today", 1), ("Tennis match tonight", 1),
    ("She runs every day", 1), ("Let us play basketball", 1),
    ("Go for the gold", 1),
    ("That was so funny", 2), ("You made me laugh", 2),
    ("Absolutely hilarious", 2), ("I cannot stop giggling", 2),
    ("Funniest thing ever", 2), ("He is such a clown", 2),
    ("Laugh out loud moment", 2), ("Tears from laughing", 2),
    ("Comedy at its best", 2), ("Pure comedy", 2), ("So ridiculous", 2),
    ("Not feeling happy", 3), ("I feel sad", 3), ("That is awful", 3),
    ("I am so disappointed", 3), ("Nothing is working out", 3),
    ("Feeling blue today", 3), ("Really struggling right now", 3),
    ("I feel so alone", 3), ("Terrible news today", 3),
    ("Everything feels wrong", 3), ("Hard day today", 3),
    ("Lets eat dinner", 4), ("I am starving", 4),
    ("Food is ready", 4), ("Great food here", 4),
    ("Time for breakfast", 4), ("Grab a snack", 4),
    ("The chef is amazing", 4), ("Tastes incredible", 4),
    ("Meal prep Sunday", 4), ("Home cooked goodness", 4),
    ("Dinner is served", 4),
]

X_train_raw = [text  for text, _ in TRAIN_DATA]
y_train     = np.array([label for _, label in TRAIN_DATA])
X_test_raw  = [text  for text, _ in TEST_DATA]
y_test      = np.array([label for _, label in TEST_DATA])

print(f'Train: {len(X_train_raw)} sentences')
print(f'Test : {len(X_test_raw)} sentences')
print()
print('Sample training data:')
for i in [0, 30, 60, 90, 120]:
    print(f'  {LABEL_EMOJIS[y_train[i]]}  {X_train_raw[i]}')


### 1.2 — Load GloVe via gensim


In [ ]:
import gensim.downloader as gensim_api

_GENSIM_NAMES = {50: 'glove-wiki-gigaword-50',
                 100: 'glove-wiki-gigaword-100',
                 300: 'glove-wiki-gigaword-300'}

print(f'Loading GloVe {GLOVE_DIM}d via gensim (first run downloads, then cached)...')
_gm = gensim_api.load(_GENSIM_NAMES[GLOVE_DIM])
glove = {word: _gm[word] for word in _gm.index_to_key}
print(f'Loaded {len(glove):,} vectors of dim {GLOVE_DIM}')


### 1.3 — Implement `build_vocab_and_index`


In [ ]:
def build_vocab_and_index(
    sentences: list[str],
) -> tuple[dict, dict]:
    """
    Build a word-to-index mapping from a list of sentences.

    Tokenisation: lowercase and split on whitespace.
    Index 0 is reserved for padding.
    Words are indexed starting from 1.

    Parameters
    ----------
    sentences : list of str

    Returns
    -------
    word2idx : dict
        word → integer index (1-based).
    idx2word : dict
        integer index → word.
    """
    # YOUR CODE HERE
    raise NotImplementedError


word2idx, idx2word = build_vocab_and_index(X_train_raw + X_test_raw)
VOCAB_SIZE = len(word2idx) + 1   # +1 for padding index 0
print(f'Vocabulary size: {VOCAB_SIZE}')
print(f'Sample entries: {list(word2idx.items())[:8]}')


### 1.4 — Implement `sentences_to_indices`


In [ ]:
def sentences_to_indices(
    sentences: list[str],
    word2idx: dict,
    max_len: int,
) -> np.ndarray:
    """
    Convert sentences to a padded matrix of word indices.

    Words not in word2idx are mapped to index 0 (padding).
    Sequences longer than max_len are truncated on the right.
    Sequences shorter than max_len are post-padded with zeros.

    Parameters
    ----------
    sentences : list of str
    word2idx : dict
    max_len : int

    Returns
    -------
    np.ndarray, shape (len(sentences), max_len)
        Integer matrix of word indices.

    Examples
    --------
    >>> sentences_to_indices(['I love you'], word2idx, max_len=5)
    array([[idx_i, idx_love, idx_you, 0, 0]])
    """
    # YOUR CODE HERE
    raise NotImplementedError


X_train_idx = sentences_to_indices(X_train_raw, word2idx, MAX_LEN)
X_test_idx  = sentences_to_indices(X_test_raw,  word2idx, MAX_LEN)

print(f'X_train_idx shape: {X_train_idx.shape}')
print(f'X_test_idx  shape: {X_test_idx.shape}')
print()
print(f'Example: "{X_train_raw[0]}"')
print(f'Indices: {X_train_idx[0]}')


---
## Part 2 — Emojifier-V1: GloVe Mean Pool

This is essentially what you built in TP-04 — mean pooling of GloVe vectors,
then a softmax classifier. No word order.

It is the baseline you will compare against V2.


### 2.1 — Implement `mean_pool_sentences`


In [ ]:
def mean_pool_sentences(
    sentences: list[str],
    glove: dict,
    dim: int = 50,
) -> np.ndarray:
    """
    Compute mean-pooled GloVe embeddings for a list of sentences.

    Parameters
    ----------
    sentences : list of str
    glove : dict
        word → np.ndarray mapping.
    dim : int, optional
        Embedding dimension, by default 50.

    Returns
    -------
    np.ndarray, shape (len(sentences), dim)
    """
    # YOUR CODE HERE
    raise NotImplementedError


X_train_v1 = mean_pool_sentences(X_train_raw, glove, GLOVE_DIM)
X_test_v1  = mean_pool_sentences(X_test_raw,  glove, GLOVE_DIM)
print(f'X_train_v1: {X_train_v1.shape}')


### 2.2 — Build and train Emojifier-V1


In [ ]:
def build_emojifier_v1(
    input_dim: int,
    num_classes: int,
    learning_rate: float = 0.01,
) -> tf.keras.Model:
    """
    Build Emojifier-V1: a single Dense softmax classifier on mean-pooled GloVe.

    Architecture
    ------------
    Input(input_dim,)
    Dense(num_classes, activation='softmax')

    Parameters
    ----------
    input_dim : int
        GloVe embedding dimension.
    num_classes : int
        Number of emoji classes.
    learning_rate : float, optional

    Returns
    -------
    tf.keras.Model
    """
    # YOUR CODE HERE
    raise NotImplementedError


In [ ]:
# One-hot encode labels for V1 (categorical cross-entropy)
y_train_oh = tf.keras.utils.to_categorical(y_train, NUM_CLASSES)
y_test_oh  = tf.keras.utils.to_categorical(y_test,  NUM_CLASSES)

v1_model = build_emojifier_v1(GLOVE_DIM, NUM_CLASSES)
v1_model.summary()

history_v1 = v1_model.fit(
    X_train_v1, y_train_oh,
    epochs=EPOCHS,
    batch_size=BATCH_SIZE,
    validation_data=(X_test_v1, y_test_oh),
    verbose=0,
)
print(f'V1 final val accuracy: {history_v1.history["val_accuracy"][-1]:.4f}')


---
## Part 3 — Emojifier-V2: Frozen GloVe + LSTM

### 3.1 — Build the embedding matrix from GloVe

For V2 the `Embedding` layer is **initialised with GloVe weights** and **frozen**
(its weights do not change during training). Only the LSTM and Dense layers are trained.

This is classic **transfer learning**: use pre-trained representations,
train only the task-specific layers on top.


In [ ]:
def build_embedding_matrix(
    word2idx: dict,
    glove: dict,
    dim: int = 50,
) -> np.ndarray:
    """
    Build an embedding weight matrix for Keras, initialised from GloVe.

    Row i of the matrix contains the GloVe vector for the word with index i.
    Row 0 (padding index) is always a zero vector.
    Words not in GloVe are left as zeros.

    Parameters
    ----------
    word2idx : dict
        word → index mapping (1-based, 0 reserved for padding).
    glove : dict
        word → np.ndarray GloVe vectors.
    dim : int, optional
        Embedding dimension, by default 50.

    Returns
    -------
    np.ndarray, shape (len(word2idx) + 1, dim)
        Embedding weight matrix.
    """
    # YOUR CODE HERE
    raise NotImplementedError


emb_matrix = build_embedding_matrix(word2idx, glove, GLOVE_DIM)
print(f'Embedding matrix shape: {emb_matrix.shape}')

# Check: the vector for 'love' should match GloVe
love_idx = word2idx.get('love')
if love_idx:
    match = np.allclose(emb_matrix[love_idx], glove['love'])
    print(f"emb_matrix['love'] matches GloVe: {match}")


### 3.2 — Implement `build_emojifier_v2`


In [ ]:
def build_emojifier_v2(
    embedding_matrix: np.ndarray,
    max_len: int,
    num_classes: int,
    lstm_units: int = 32,
    dropout: float = 0.5,
    learning_rate: float = 0.005,
) -> tf.keras.Model:
    """
    Build Emojifier-V2: frozen GloVe embedding + LSTM.

    Architecture
    ------------
    Input(max_len,)
    Embedding(vocab_size, dim, weights=[embedding_matrix], trainable=False)
    LSTM(lstm_units, return_sequences=False)
    Dropout(dropout)
    Dense(num_classes, activation='softmax')

    Parameters
    ----------
    embedding_matrix : np.ndarray, shape (vocab_size, dim)
        Pre-computed GloVe weight matrix (from build_embedding_matrix).
    max_len : int
        Input sequence length.
    num_classes : int
        Number of output classes.
    lstm_units : int, optional
        LSTM hidden units, by default 32.
    dropout : float, optional
        Dropout rate after LSTM, by default 0.5.
    learning_rate : float, optional

    Returns
    -------
    tf.keras.Model

    Notes
    -----
    The Embedding layer must be created with trainable=False.
    This freezes the GloVe weights so only LSTM and Dense are updated.
    """
    # YOUR CODE HERE
    raise NotImplementedError


In [ ]:
v2_model = build_emojifier_v2(emb_matrix, MAX_LEN, NUM_CLASSES)
v2_model.summary()
print()
# Verify the embedding layer is frozen
emb_layer = v2_model.layers[1]
print(f'Embedding layer trainable: {emb_layer.trainable}   (should be False)')
total_params     = v2_model.count_params()
trainable_params = sum(np.prod(w.shape) for w in v2_model.trainable_weights)
print(f'Total params: {total_params:,}   Trainable: {trainable_params:,}')


In [ ]:
history_v2 = v2_model.fit(
    X_train_idx, y_train_oh,
    epochs=EPOCHS,
    batch_size=BATCH_SIZE,
    validation_data=(X_test_idx, y_test_oh),
    verbose=0,
)
print(f'V2 final val accuracy: {history_v2.history["val_accuracy"][-1]:.4f}')


---
## Part 4 — Compare V1 vs V2


In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 4))
for ax, metric in zip(axes, ['accuracy', 'loss']):
    ax.plot(history_v1.history[f'val_{metric}'], label='V1 (mean pool)', color='#e8c47a')
    ax.plot(history_v2.history[f'val_{metric}'], label='V2 (LSTM)', color='#56c8ba')
    ax.set_title(f'Validation {metric}', color='#e6edf3')
    ax.set_xlabel('Epoch')
    ax.legend()
    ax.set_facecolor('#161b22')
    ax.tick_params(colors='#8b949e')
    ax.spines[:].set_color('#30363d')
fig.patch.set_facecolor('#0d1117')
plt.tight_layout()
plt.show()


In [ ]:
label_names = [LABEL_EMOJIS[i] for i in range(NUM_CLASSES)]

for name, model, X_test_input in [
    ('V1 (mean pool)', v1_model, X_test_v1),
    ('V2 (LSTM)',      v2_model, X_test_idx),
]:
    y_pred = np.argmax(model.predict(X_test_input, verbose=0), axis=1)
    print(f'\n=== {name} ===')
    print(classification_report(y_test, y_pred, target_names=label_names))

    cm = confusion_matrix(y_test, y_pred)
    fig, ax = plt.subplots(figsize=(5, 4))
    ConfusionMatrixDisplay(cm, display_labels=label_names).plot(
        ax=ax, colorbar=False, cmap='Blues'
    )
    ax.set_title(f'{name}')
    plt.tight_layout()
    plt.show()


**📝 Your analysis — Part 4**

1. Which model performs better overall? Is the difference large or small?
2. Look at the confusion matrices. Which emoji classes are most often confused with each other? Why?
3. V1 uses mean pooling — it cannot distinguish "not happy" from "happy". Test both models on this sentence. Which one gets it right? Why?
4. V2 has frozen embeddings — only the LSTM and Dense layers are trained. What are the advantages of freezing the GloVe weights on a small dataset like this?

*Write your answers here.*


In [ ]:
# The key test: does V2 handle negation better than V1?
negation_sentences = [
    'not feeling happy',
    'not funny at all',
    'I love you',
    'I do not love you',
]

print('Negation test — V1 (mean pool) vs V2 (LSTM):')
print(f'{"Sentence":<30} {"V1":<6} {"V2":<6}')
print('-' * 44)

for sentence in negation_sentences:
    # V1: mean pool
    tokens = sentence.lower().split()
    vecs   = [glove[t] for t in tokens if t in glove]
    if vecs:
        avg = np.mean(vecs, axis=0)
    else:
        avg = np.zeros(GLOVE_DIM)
    pred_v1 = np.argmax(v1_model.predict(avg.reshape(1, -1), verbose=0))

    # V2: index sequence
    idx_seq = sentences_to_indices([sentence], word2idx, MAX_LEN)
    pred_v2 = np.argmax(v2_model.predict(idx_seq, verbose=0))

    print(f'{sentence:<30} {LABEL_EMOJIS[pred_v1]:<6} {LABEL_EMOJIS[pred_v2]:<6}')


---
## Part 5 (Bonus) — Test on Your Own Sentences

Write a few sentences of your own and see what the models predict.
Focus on tricky cases: negation, sarcasm, ambiguous sentences.


In [ ]:
MY_SENTENCES = [
    # ← add your own sentences here
    'I am absolutely not hungry',
    'That goal was not great',
    'I laughed but it was not funny',
    'We won but I feel terrible',
    'What a beautiful meal',
]

print(f'{"Sentence":<40} {"V1":<6} {"V2":<6}')
print('-' * 54)

for sentence in MY_SENTENCES:
    tokens = sentence.lower().split()
    vecs   = [glove[t] for t in tokens if t in glove]
    avg    = np.mean(vecs, axis=0) if vecs else np.zeros(GLOVE_DIM)
    pred_v1 = np.argmax(v1_model.predict(avg.reshape(1, -1), verbose=0))

    idx_seq = sentences_to_indices([sentence], word2idx, MAX_LEN)
    pred_v2 = np.argmax(v2_model.predict(idx_seq, verbose=0))

    print(f'{sentence:<40} {LABEL_EMOJIS[pred_v1]:<6} {LABEL_EMOJIS[pred_v2]:<6}')


---
## Summary


In [ ]:
print('TP 05-B — Emojifier results')
print(f'Train: {len(X_train_raw)} sentences  Test: {len(X_test_raw)} sentences')
print(f'GloVe: {GLOVE_DIM}d   MAX_LEN: {MAX_LEN}   EPOCHS: {EPOCHS}')
print()
from sklearn.metrics import f1_score
for name, model, X_in in [
    ('V1 mean pool', v1_model, X_test_v1),
    ('V2 LSTM',      v2_model, X_test_idx),
]:
    y_pred = np.argmax(model.predict(X_in, verbose=0), axis=1)
    f1 = f1_score(y_test, y_pred, average='macro')
    print(f'{name:<20} Test Macro F1: {f1:.4f}')
print()
print('What is next: Session 06 — Transformers and self-attention')
